# 07 — Severity Modelling

## Main question

**Among incidents with reported aircraft damage, can damage severity be modelled reliably enough to retain this secondary component for downstream simulation?**

This notebook implements the project's second-stage severity model. It is **conditional on a wildlife strike already being reported and on `INDICATED_DAMAGE = 1`**. It does not predict whether a strike will occur and it does not replace the primary binary damage-probability system developed in Notebooks 05–06.

The notebook follows the same modelling principles used in Notebook 05:

- scenario-eligible predictors only;
- explicit leakage checks;
- an expanding chronological cross-validation design inside 1990–2018;
- a majority-class baseline;
- Logistic Regression, Random Forest, and XGBoost candidates;
- hyperparameter tuning without touching the final test;
- conventional classification metrics **and** probability-quality metrics;
- reproducible seeds and persistent candidate outputs.

Severity adds several requirements:

1. `M?` is retained as its own class because the FAA documentation defines it as **undetermined level**, not minor damage.
2. `D` (Destroyed) is merged with `S` (Substantial) **for modelling only** because the Destroyed class is too sparse for stable independent estimation and validation.
3. The resulting target is a **nominal three-class outcome**: `M`, `M?`, and `S+D`. `M?` is not placed on an ordinal scale.
4. **Macro-F1** is the primary hyperparameter-selection metric so that minority classes cannot be hidden by the dominant class.
5. Precision, recall, and F1 for the `S+D` class are always reported separately.
6. Class weighting is **tested rather than imposed**. Logistic Regression and Random Forest use their native `class_weight` support; multiclass XGBoost uses training-row `sample_weight`.
7. Calibration and temporal validation are required before severity can be passed to simulation.
8. The notebook ends with an explicit **RETAIN / SIMPLIFY / DROP** decision gate. A decision not to use severity in simulation is an acceptable scientific result.

> **Execution note:** computationally expensive tuning cells are intentionally left unexecuted in this review copy. Markdown interpretation cells contain clearly marked placeholders to be replaced after the notebook is run on the project machine.

## 1. Methodological decisions and academic justification

### 1.1 FAA severity definitions

The original FAA data dictionary identifies the damage field as a level selected by the Database Manager and provides ICAO-based definitions:

- **M — Minor:** the aircraft can be rendered airworthy through simple repairs or replacements without an extensive inspection.
- **M? — Undetermined level:** the aircraft was damaged, but the available details are insufficient to determine the extent.
- **S — Substantial:** damage or structural failure adversely affects structural strength, performance, or flight characteristics and normally requires major repair or replacement of the affected component.
- **D — Destroyed:** the damage makes restoration to an airworthy condition inadvisable.

Because `M?` represents **uncertainty about severity**, it is not a midpoint between `M` and `S`, and it must not be silently recoded as Minor.

### 1.2 Why `D` is merged with `S` for modelling

Destroyed and Substantial remain **distinct aviation-damage concepts**. Their merger here does **not** claim that they have equal physical, operational, financial, or safety consequences.

The modelling dataset contains very few Destroyed observations relative to the other severity categories. A class with only a handful of observations in validation or final testing would yield extremely unstable class-specific recall, precision, calibration, and probability estimates. Therefore, Destroyed observations are preserved in the raw target but combined with Substantial as a **severe modelling category (`S+D`)**.

This is a statistical-support decision, not a redefinition of aviation severity. It also avoids discarding Destroyed incidents entirely and keeps them represented at the severe end of the modelled outcome.

### 1.3 Why Macro-F1 is the primary tuning metric

Severity is an imbalanced multiclass problem. Overall accuracy and weighted metrics can be dominated by common classes. Macro-F1 calculates F1 separately for each class and then gives each class equal influence in the average. It is therefore used as the primary tuning criterion.

Macro-F1 is **not used alone**. The notebook also reports:

- class-specific precision, recall, and F1;
- macro precision and macro recall;
- weighted F1;
- balanced accuracy;
- confusion matrices;
- multiclass log loss;
- multiclass Brier score;
- macro one-vs-rest average precision;
- macro one-vs-rest ROC-AUC where all classes are evaluable.

The `S+D` precision, recall, and F1 are explicitly retained in every important comparison table.

### 1.4 Class weighting

Imbalanced-learning literature supports cost-sensitive/class-weighted learning as a standard response to unequal class frequencies, but probability-oriented research also cautions that imbalance corrections can distort calibration when imposed without validation.

For that reason, weighting is a **hyperparameter choice**:

- Logistic Regression: `class_weight ∈ {None, "balanced"}`;
- Random Forest: `class_weight ∈ {None, "balanced_subsample"}`;
- XGBoost: `sample_weight_mode ∈ {"none", "balanced"}` using fold-specific balanced observation weights.

No SMOTE or synthetic oversampling is used in the primary experiment. This keeps the workflow simpler, avoids generating synthetic combinations of scenario categories, and lets calibration be evaluated honestly afterward.

### References supporting these choices

- FAA Wildlife Strike Database data dictionary / field definitions; severity definitions reproduce the ICAO-based categories included in the project source dictionary.
- He, H., & Garcia, E. A. (2009). *Learning from Imbalanced Data*. IEEE Transactions on Knowledge and Data Engineering, 21(9), 1263–1284.
- Pedregosa, F., et al. (2011). *Scikit-learn: Machine Learning in Python*. Journal of Machine Learning Research, 12, 2825–2830.
- Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System*. Proceedings of KDD.
- van den Goorbergh, R., van Smeden, M., Timmerman, D., & Van Calster, B. (2022/2023). Work examining how class-imbalance corrections can affect risk-prediction calibration. This motivates **testing** weighting rather than assuming it must improve the final probability system.

## 2. Environment and reproducibility

`xgboost` is used to retain the same candidate-model family as Notebook 05. If it is unavailable in the active environment, install it once and rerun the imports.

In [1]:
# Uncomment only if xgboost is missing in the active environment.
# %pip install -q xgboost

In [2]:
from pathlib import Path
import json
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import ParameterGrid
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    log_loss,
    average_precision_score,
    roc_auc_score,
)

from xgboost import XGBClassifier

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 200)

SEED = 42
np.random.seed(SEED)

CLASS_LABELS = ["M", "M?", "S+D"]
LABEL_TO_INT = {label: i for i, label in enumerate(CLASS_LABELS)}
INT_TO_LABEL = {i: label for label, i in LABEL_TO_INT.items()}
N_CLASSES = len(CLASS_LABELS)

print("Seed:", SEED)
print("Severity classes:", CLASS_LABELS)

Seed: 42
Severity classes: ['M', 'M?', 'S+D']


## 3. Portable project paths

The notebook searches upward for a repository containing `data`. In the final repository, the canonical source should be:

`data/processed/faa_strikes_analytical.csv`

Persistent outputs are separated into:

- `outputs/severity/` — tables, predictions, CV results, calibration evidence;
- `models/candidates/` — fitted severity candidates;
- `models/final/` — only a severity artifact that passes the final decision gate.

This preserves the same candidates-versus-final distinction used by the modelling workflow.

In [3]:
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data").exists() or (candidate / "models").exists():
        ROOT = candidate
        break

DATA_CANDIDATES = [
    ROOT / "data" / "processed" / "faa_strikes_analytical.csv",
    ROOT / "data" / "processed" / "faa_strikes_analytical(3).csv",
    Path("/mnt/data/faa_strikes_analytical(3).csv"),
]

DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not locate faa_strikes_analytical.csv. "
        "Update DATA_CANDIDATES to point to the canonical Notebook 02 output."
    )

FEATURE_LIST_CANDIDATES = [
    ROOT / "outputs" / "feature_lists.json",
    ROOT / "data" / "processed" / "feature_lists.json",
    ROOT / "docs" / "feature_lists.json",
]
FEATURE_LIST_PATH = next((p for p in FEATURE_LIST_CANDIDATES if p.exists()), None)

OUTPUT_DIR = ROOT / "outputs" / "severity"
CANDIDATE_MODEL_DIR = ROOT / "models" / "candidates"
FINAL_MODEL_DIR = ROOT / "models" / "final"

for folder in [OUTPUT_DIR, CANDIDATE_MODEL_DIR, FINAL_MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("DATA_PATH:", DATA_PATH)
print("FEATURE_LIST_PATH:", FEATURE_LIST_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CANDIDATE_MODEL_DIR:", CANDIDATE_MODEL_DIR)
print("FINAL_MODEL_DIR:", FINAL_MODEL_DIR)

ROOT: c:\Users\davis\Programing\MDA_programs\Capstone\faa-wildlife-strike-damage-analysis
DATA_PATH: c:\Users\davis\Programing\MDA_programs\Capstone\faa-wildlife-strike-damage-analysis\data\processed\faa_strikes_analytical.csv
FEATURE_LIST_PATH: c:\Users\davis\Programing\MDA_programs\Capstone\faa-wildlife-strike-damage-analysis\docs\feature_lists.json
OUTPUT_DIR: c:\Users\davis\Programing\MDA_programs\Capstone\faa-wildlife-strike-damage-analysis\outputs\severity
CANDIDATE_MODEL_DIR: c:\Users\davis\Programing\MDA_programs\Capstone\faa-wildlife-strike-damage-analysis\models\candidates
FINAL_MODEL_DIR: c:\Users\davis\Programing\MDA_programs\Capstone\faa-wildlife-strike-damage-analysis\models\final


## 4. Load the canonical analytical dataset

Notebook 07 consumes the cleaned analytical dataset rather than redefining cleaning rules. The raw `DAMAGE_LEVEL` field is retained for auditing, while a new modelling target is created locally.

In [4]:
df = pd.read_csv(DATA_PATH, low_memory=False)

required_columns = {
    "INDICATED_DAMAGE", "DAMAGE_LEVEL", "INCIDENT_YEAR", "INDEX_NR"
}
missing_required = sorted(required_columns - set(df.columns))
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(
    "Year coverage:",
    int(df["INCIDENT_YEAR"].min()),
    "to",
    int(df["INCIDENT_YEAR"].max()),
)

Rows: 319,107
Columns: 150
Year coverage: 1990 to 2024


## 5. Severity feasibility audit before modelling

The checklist requires the severity component to justify its own feasibility before any model is trained.

This section checks:

- restriction to damaged incidents;
- raw severity classes and missingness;
- consistency with the binary damage target;
- class counts by chronological split;
- the amount of support available for `D`;
- whether the `M?` frequency changes strongly over time.

A sharp temporal change in `M?` must be interpreted as a possible reporting/completeness shift. It must **not** be interpreted as an ordered movement toward more or less physical damage.

In [5]:
damaged = df.loc[df["INDICATED_DAMAGE"].eq(1)].copy()

severity_audit = pd.DataFrame({
    "measure": [
        "All analytical incidents",
        "Damaged incidents",
        "Damaged incidents with missing DAMAGE_LEVEL",
        "Damaged incidents with non-missing DAMAGE_LEVEL",
    ],
    "count": [
        len(df),
        len(damaged),
        int(damaged["DAMAGE_LEVEL"].isna().sum()),
        int(damaged["DAMAGE_LEVEL"].notna().sum()),
    ],
})
display(severity_audit)

raw_class_counts = (
    damaged["DAMAGE_LEVEL"]
    .value_counts(dropna=False)
    .rename_axis("DAMAGE_LEVEL")
    .reset_index(name="count")
)
raw_class_counts["proportion"] = raw_class_counts["count"] / len(damaged)
display(raw_class_counts)

raw_class_counts.to_csv(
    OUTPUT_DIR / "severity_raw_class_counts.csv", index=False
)

,measure,count
0,All analytical incidents,319107
1,Damaged incidents,20908
2,Damaged incidents with missing DAMAGE_LEVEL,0
3,Damaged incidents with non-missing DAMAGE_LEVEL,20908


,DAMAGE_LEVEL,count,proportion
0,M,8657,0.414052
1,M?,7818,0.373924
2,S,4344,0.207767
3,D,89,0.004257


### Interpretation — raw severity feasibility

The analytical dataset contains 20,908 damaged incidents, and all of them have a recorded `DAMAGE_LEVEL`; therefore, no damaged observations are lost because of missing severity labels. Minor damage (`M`) is the largest raw category with 8,657 cases (41.4%), followed by undetermined severity (`M?`) with 7,818 (37.4%), Substantial damage (`S`) with 4,344 (20.8%), and Destroyed (`D`) with only 89 cases (0.43%). The `D` category is therefore too sparse to support reliable independent training, calibration, and later-period validation. In accordance with the modelling decision established above, `D` is retained in the original data but combined with `S` into the `S+D` severe modelling class. This aggregation is based on statistical support and does not imply that Substantial and Destroyed damage are operationally equivalent. The large number of `M?` cases is also retained as a separate outcome because it represents an undetermined damage level rather than Minor damage.

In [6]:
# Preserve the raw label and create the modelling label.
damaged["DAMAGE_LEVEL_RAW_MODEL_AUDIT"] = damaged["DAMAGE_LEVEL"]

def map_severity_for_model(value):
    if value == "M":
        return "M"
    if value == "M?":
        return "M?"
    if value in {"S", "D"}:
        return "S+D"
    return np.nan

damaged["SEVERITY_MODEL"] = damaged["DAMAGE_LEVEL"].map(map_severity_for_model)

# Only records with one of the documented damaged severity codes can enter the model.
severity_df = damaged.loc[damaged["SEVERITY_MODEL"].notna()].copy()
severity_df["SEVERITY_TARGET"] = severity_df["SEVERITY_MODEL"].map(LABEL_TO_INT).astype(int)

print("Rows retained for severity modelling:", f"{len(severity_df):,}")
display(
    severity_df[["DAMAGE_LEVEL", "SEVERITY_MODEL"]]
    .value_counts()
    .rename("count")
    .reset_index()
)

assert set(severity_df["DAMAGE_LEVEL"].unique()).issubset({"M", "M?", "S", "D"})
assert set(severity_df["SEVERITY_MODEL"].unique()) == set(CLASS_LABELS)

Rows retained for severity modelling: 20,908


,DAMAGE_LEVEL,SEVERITY_MODEL,count
0,M,M,8657
1,M?,M?,7818
2,S,S+D,4344
3,D,S+D,89


### Decision recorded in the notebook: `D → S+D`

`D` is merged with `S` **only in `SEVERITY_MODEL`**. The original `DAMAGE_LEVEL` and the copied audit field remain unchanged.

The decision is justified by support, not semantics. Destroyed damage remains more severe than Substantial damage under the FAA/ICAO definitions. However, reliable standalone estimation requires enough observations in training **and** enough observations in later validation/test periods to estimate errors and calibration with tolerable stability. When only a very small number of Destroyed cases occur in those periods, a separate `D` class would produce fragile metrics.

Combining `D` with `S` retains Destroyed incidents in the severe outcome while avoiding a misleading appearance of precision.

## 6. Locked chronological partitions

The same periods used in Notebooks 05–06 are retained:

- **Training / tuning:** 1990–2018
- **Validation / model-selection and calibration development:** 2019–2021
- **Locked final test:** 2022–2024

The final test must remain unopened until the model family, hyperparameters, calibration method, and decision criteria have been selected from earlier periods.

In [7]:
train_mask = severity_df["INCIDENT_YEAR"].between(1990, 2018)
validation_mask = severity_df["INCIDENT_YEAR"].between(2019, 2021)
test_mask = severity_df["INCIDENT_YEAR"].between(2022, 2024)

split_summary = []
for split_name, mask in [
    ("train_1990_2018", train_mask),
    ("validation_2019_2021", validation_mask),
    ("test_2022_2024_LOCKED", test_mask),
]:
    subset = severity_df.loc[mask]
    counts = subset["SEVERITY_MODEL"].value_counts()
    row = {
        "split": split_name,
        "rows": len(subset),
    }
    for label in CLASS_LABELS:
        row[f"count_{label}"] = int(counts.get(label, 0))
        row[f"rate_{label}"] = float(counts.get(label, 0) / len(subset)) if len(subset) else np.nan
    split_summary.append(row)

split_summary = pd.DataFrame(split_summary)
display(split_summary)
split_summary.to_csv(OUTPUT_DIR / "severity_split_class_counts.csv", index=False)

,split,rows,count_M,rate_M,count_M?,rate_M?,count_S+D,rate_S+D
0,train_1990_2018,16737,8178,0.488618,4407,0.263309,4152,0.248073
1,validation_2019_2021,1925,209,0.108571,1587,0.824416,129,0.067013
2,test_2022_2024_LOCKED,2246,270,0.120214,1824,0.812110,152,0.067676


### Interpretation — temporal class structure

The severity distribution changes substantially across the chronological partitions. In the 1990–2018 training period, `M` is the largest class with 8,178 of 16,737 cases (48.9%), while `M?` accounts for 4,407 (26.3%) and `S+D` for 4,152 (24.8%). In contrast, `M?` becomes dominant in both later periods: it represents 1,587 of 1,925 validation incidents (82.4%) in 2019–2021 and 1,824 of 2,246 final-test incidents (81.2%) in 2022–2024. Over the same periods, `M` falls to approximately 10.9% and 12.0%, while `S+D` falls to approximately 6.7% in both periods. This represents a major temporal distribution shift in the recorded severity outcome. Because `M?` means that the level of damage could not be determined, this change should not be interpreted as evidence that physical aircraft damage became less severe. Instead, it indicates a substantial change in severity reporting or completeness across time and creates an important generalization challenge for models trained mainly on earlier records.

## 7. Canonical feature lists and leakage protection

Severity is conditional on damage, but that does **not** make post-event damage information acceptable as predictors.

The model uses the same scenario-eligible predictors approved for Notebook 05. Fields describing the realized damage, component damage, injuries/fatalities, repair costs, operational consequences, or administrative/reporting information remain excluded.

This is essential because Notebook 10 can only ask the severity model about inputs known in the simulated strike scenario.

In [8]:
fallback_feature_lists = {
    "core_features": [
        "SEASON", "MONTH_SIN", "MONTH_COS", "WILDLIFE_TYPE", "SIZE",
        "NUM_STRUCK", "AC_CLASS", "AC_MASS_GROUP", "TYPE_ENG",
        "NUM_ENGS", "WARNED"
    ],
    "extended_features": [
        "SEASON", "MONTH_SIN", "MONTH_COS", "WILDLIFE_TYPE", "SIZE",
        "NUM_STRUCK", "AC_CLASS", "AC_MASS_GROUP", "TYPE_ENG",
        "NUM_ENGS", "WARNED", "PHASE_OF_FLIGHT", "HEIGHT", "SPEED",
        "TIME_OF_DAY", "SKY", "PRECIPITATION", "STATE", "FAAREGION"
    ],
    "airport_aware_features": [
        "SEASON", "MONTH_SIN", "MONTH_COS", "WILDLIFE_TYPE", "SIZE",
        "NUM_STRUCK", "AC_CLASS", "AC_MASS_GROUP", "TYPE_ENG",
        "NUM_ENGS", "WARNED", "PHASE_OF_FLIGHT", "HEIGHT", "SPEED",
        "TIME_OF_DAY", "SKY", "PRECIPITATION", "STATE", "FAAREGION",
        "AIRPORT_ID"
    ],
    "seed": SEED,
}

if FEATURE_LIST_PATH is not None:
    with open(FEATURE_LIST_PATH, "r", encoding="utf-8") as f:
        feature_lists = json.load(f)
else:
    feature_lists = fallback_feature_lists.copy()
    warnings.warn(
        "feature_lists.json was not found. The approved Notebook 02/05 fallback "
        "is being used. Restore the canonical JSON artifact before final submission."
    )

EXTENDED_FEATURES = [
    c for c in feature_lists["extended_features"] if c in severity_df.columns
]
AIRPORT_AWARE_FEATURES = [
    c for c in feature_lists["airport_aware_features"] if c in severity_df.columns
]

FEATURE_SETS = {
    "extended_geographic": EXTENDED_FEATURES,
    "airport_aware": AIRPORT_AWARE_FEATURES,
}

forbidden_predictors = {
    "DAMAGE_LEVEL", "DAMAGE_LEVEL_RAW", "DAMAGE_LEVEL_RAW_MODEL_AUDIT",
    "SEVERITY_MODEL", "SEVERITY_TARGET",
    "INDICATED_DAMAGE",
    "EFFECT", "EFFECT_OTHER", "AOS",
    "COST_REPAIRS", "COST_OTHER",
    "COST_REPAIRS_INFL_ADJ", "COST_OTHER_INFL_ADJ",
    "NR_INJURIES", "NR_FATALITIES",
    "INCIDENT_YEAR", "TEMPORAL_SPLIT", "INDEX_NR",
    "TARGET_CONFLICT_FLAG", "SEVERITY_ELIGIBLE_FLAG",
}
forbidden_predictors |= {
    c for c in severity_df.columns if c.startswith(("DAM_", "STR_", "ING_"))
}

for set_name, features in FEATURE_SETS.items():
    leaked = sorted(set(features) & forbidden_predictors)
    assert not leaked, f"{set_name} contains forbidden predictors: {leaked}"

feature_set_table = pd.DataFrame([
    {
        "feature_set": name,
        "feature_count": len(features),
        "includes_airport_id": "AIRPORT_ID" in features,
        "leakage_check": "PASS",
    }
    for name, features in FEATURE_SETS.items()
])
display(feature_set_table)

print("Leakage assertions passed.")

,feature_set,feature_count,includes_airport_id,leakage_check
0,extended_geographic,19,False,PASS
1,airport_aware,20,True,PASS


Leakage assertions passed.


## 8. Training-only preprocessing

Preprocessing mirrors Notebook 05:

- numerical values: training-fold median imputation;
- Logistic Regression: numerical scaling;
- `NUM_STRUCK`: ordered encoding of documented count ranges;
- nominal categories: explicit missing label plus sparse one-hot encoding;
- rare categories: training-fitted grouping through `min_frequency`.

The use of sparse encoding is particularly useful for Random Forest runtime because the airport-aware feature set can produce many one-hot columns. No learned preprocessing is fitted globally.

In [9]:
NUM_STRUCK_ORDER = [["1", "2–10", "11–100", "More than 100"]]

def make_preprocessor(features, scale_numeric=False, min_category_frequency=50):
    ordinal_features = [c for c in ["NUM_STRUCK"] if c in features]

    numeric_features = [
        c for c in features
        if c != "NUM_STRUCK" and pd.api.types.is_numeric_dtype(severity_df[c])
    ]

    categorical_features = [
        c for c in features
        if c not in ordinal_features + numeric_features
    ]

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        # with_mean=False preserves sparse compatibility after column concatenation.
        numeric_steps.append(("scaler", StandardScaler(with_mean=False)))

    transformers = []

    if numeric_features:
        transformers.append((
            "numeric",
            Pipeline(numeric_steps),
            numeric_features,
        ))

    if ordinal_features:
        transformers.append((
            "num_struck_ordinal",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ordinal", OrdinalEncoder(
                    categories=NUM_STRUCK_ORDER,
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                    encoded_missing_value=-1,
                )),
            ]),
            ordinal_features,
        ))

    if categorical_features:
        transformers.append((
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(
                    strategy="constant",
                    fill_value="Not reported",
                )),
                ("onehot", OneHotEncoder(
                    handle_unknown="infrequent_if_exist",
                    min_frequency=min_category_frequency,
                    sparse_output=True,
                )),
            ]),
            categorical_features,
        ))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )

## 9. Expanding chronological cross-validation inside 1990–2018

The fold structure is intentionally the same as Notebook 05:

| Fold | Training years | Validation years |
|---|---|---|
| 1 | 1990–2003 | 2004–2006 |
| 2 | 1990–2006 | 2007–2009 |
| 3 | 1990–2009 | 2010–2012 |
| 4 | 1990–2012 | 2013–2015 |
| 5 | 1990–2015 | 2016–2018 |

This allows hyperparameter CV while preserving temporal order. The 2019–2021 validation period and 2022–2024 final test are outside the tuning folds.

In [10]:
CV_FOLDS = [
    {"fold": 1, "train_years": range(1990, 2004), "validation_years": range(2004, 2007)},
    {"fold": 2, "train_years": range(1990, 2007), "validation_years": range(2007, 2010)},
    {"fold": 3, "train_years": range(1990, 2010), "validation_years": range(2010, 2013)},
    {"fold": 4, "train_years": range(1990, 2013), "validation_years": range(2013, 2016)},
    {"fold": 5, "train_years": range(1990, 2016), "validation_years": range(2016, 2019)},
]

cv_summary = []
for spec in CV_FOLDS:
    fold_train = severity_df.loc[
        train_mask & severity_df["INCIDENT_YEAR"].isin(spec["train_years"])
    ]
    fold_val = severity_df.loc[
        train_mask & severity_df["INCIDENT_YEAR"].isin(spec["validation_years"])
    ]

    row = {
        "fold": spec["fold"],
        "train_start": min(spec["train_years"]),
        "train_end": max(spec["train_years"]),
        "validation_start": min(spec["validation_years"]),
        "validation_end": max(spec["validation_years"]),
        "train_rows": len(fold_train),
        "validation_rows": len(fold_val),
    }

    for label in CLASS_LABELS:
        row[f"train_{label}"] = int((fold_train["SEVERITY_MODEL"] == label).sum())
        row[f"validation_{label}"] = int((fold_val["SEVERITY_MODEL"] == label).sum())

    cv_summary.append(row)

cv_summary = pd.DataFrame(cv_summary)
display(cv_summary)
cv_summary.to_csv(OUTPUT_DIR / "severity_chronological_cv_design.csv", index=False)

# Fail early if any fold cannot evaluate all three classes.
for label in CLASS_LABELS:
    assert (cv_summary[f"train_{label}"] > 0).all(), f"Training fold lacks {label}"
    assert (cv_summary[f"validation_{label}"] > 0).all(), f"Validation fold lacks {label}"

,fold,train_start,train_end,validation_start,validation_end,train_rows,validation_rows,train_M,validation_M,train_M?,validation_M?,train_S+D,validation_S+D
0,1,1990,2003,2004,2006,7577,1843,4099,917,1328,462,2150,464
1,2,1990,2006,2007,2009,9420,1703,5016,843,1790,498,2614,362
2,3,1990,2009,2010,2012,11123,1778,5859,1014,2288,282,2976,482
3,4,1990,2012,2013,2015,12901,1832,6873,880,2570,435,3458,517
4,5,1990,2015,2016,2018,14733,2004,7753,425,3005,1402,3975,177


## 10. Candidate models

Four references are retained:

1. **Dummy majority baseline** — establishes no-skill classification behavior.
2. **Multiclass Logistic Regression** — linear/interpretable baseline.
3. **Random Forest** — nonlinear bagged-tree candidate.
4. **XGBoost** — nonlinear gradient-boosting candidate.

Random Forest tuning is separated into its own cell because it was the most computationally expensive family in Notebook 05. This severity dataset is much smaller than the full binary-damage dataset, and two implementation choices reduce unnecessary runtime:

- sparse preprocessing is preserved;
- Random Forest uses all available CPU cores through `n_jobs=-1`.

The canonical grid is not reduced merely for speed. If runtime remains excessive on the project machine, an optional reproducible sampled-grid switch is provided below, but the default is the full grid.

In [11]:
def make_pipeline(model_name, features):
    if model_name == "logistic":
        model = LogisticRegression(
            max_iter=3000,
            solver="lbfgs",
            penalty="l2",
            C=1.0,
            class_weight=None,
            random_state=SEED,
        )
        prep = make_preprocessor(features, scale_numeric=True)

    elif model_name == "random_forest":
        model = RandomForestClassifier(
            n_estimators=300,
            class_weight=None,
            n_jobs=-1,
            random_state=SEED,
        )
        prep = make_preprocessor(features, scale_numeric=False)

    elif model_name == "xgboost":
        model = XGBClassifier(
            objective="multi:softprob",
            num_class=N_CLASSES,
            eval_metric="mlogloss",
            tree_method="hist",
            n_jobs=-1,
            random_state=SEED,
        )
        prep = make_preprocessor(features, scale_numeric=False)

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return Pipeline([
        ("preprocess", prep),
        ("model", model),
    ])

## 11. Hyperparameter grids

The values deliberately remain close to Notebook 05 so model capacity is comparable across the two modelling stages.

### Logistic Regression

Notebook 05 used `C = {0.1, 1, 5}`, L1/L2 regularization, and class weighting. For multiclass severity:

- `lbfgs` evaluates L2;
- `saga` evaluates L1 and L2;
- `class_weight` compares no weighting with balanced weighting.

### Random Forest

The Notebook 05 values are retained:

- 250 vs 400 trees;
- depth 12 vs unrestricted;
- split and leaf regularization;
- `sqrt` vs 50% feature sampling;
- no class weighting vs `balanced_subsample`.

### XGBoost

The Notebook 05 tree, learning-rate, sampling, and regularization values are retained. Binary `scale_pos_weight` is **not** carried into the multiclass problem. Instead, `sample_weight_mode` determines whether each training fold receives inverse-frequency balanced row weights.

This makes the weighting comparison conceptually consistent across all three model families without pretending that the binary XGBoost parameter has a direct multiclass interpretation.

In [12]:
LOGISTIC_PARAMETER_GRID = [
    {
        "model__solver": ["saga"],
        "model__penalty": ["l1", "l2"],
        "model__C": [0.1, 1.0, 5.0],
        "model__class_weight": [None, "balanced"],
    },
    {
        "model__solver": ["lbfgs"],
        "model__penalty": ["l2"],
        "model__C": [0.1, 1.0, 5.0],
        "model__class_weight": [None, "balanced"],
    },
]

RANDOM_FOREST_PARAMETER_GRID = {
    "model__n_estimators": [250, 400],
    "model__max_depth": [12, None],
    "model__min_samples_split": [2, 10],
    "model__min_samples_leaf": [2, 10],
    "model__max_features": ["sqrt", 0.5],
    "model__class_weight": [None, "balanced_subsample"],
}

XGBOOST_PARAMETER_GRID = {
    "model__n_estimators": [300, 600],
    "model__max_depth": [3, 6],
    "model__learning_rate": [0.03, 0.08],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
    "model__min_child_weight": [1, 5],
    "model__reg_lambda": [1.0, 5.0],
    "sample_weight_mode": ["none", "balanced"],
}

parameter_grids = {
    "logistic": LOGISTIC_PARAMETER_GRID,
    "random_forest": RANDOM_FOREST_PARAMETER_GRID,
    "xgboost": XGBOOST_PARAMETER_GRID,
}

MODEL_NAMES = ["logistic", "random_forest", "xgboost"]

candidate_counts = []
for model_name, grid in parameter_grids.items():
    settings_per_feature_set = len(list(ParameterGrid(grid)))
    candidate_counts.append({
        "model": model_name,
        "settings_per_feature_set": settings_per_feature_set,
        "feature_sets": len(FEATURE_SETS),
        "cv_folds": len(CV_FOLDS),
        "total_model_fits": settings_per_feature_set * len(FEATURE_SETS) * len(CV_FOLDS),
    })

candidate_count_table = pd.DataFrame(candidate_counts)
display(candidate_count_table)
print("Total chronological-CV fits:", int(candidate_count_table["total_model_fits"].sum()))

,model,settings_per_feature_set,feature_sets,cv_folds,total_model_fits
0,logistic,18,2,5,180
1,random_forest,64,2,5,640
2,xgboost,256,2,5,2560


Total chronological-CV fits: 3380


### Optional Random Forest runtime switch

The default below preserves the complete Random Forest grid. If the full grid is still impractical on the local machine, set `RF_MAX_SETTINGS` to a fixed number such as `24`. The notebook will draw a **reproducible sample of complete parameter combinations** using `SEED`.

For the final academic run, prefer `None` if runtime permits. If sampling is used, report it explicitly as randomized hyperparameter search rather than exhaustive grid search.

In [13]:
RF_MAX_SETTINGS = None  # Preferred final setting. Example faster run: 24.

def get_rf_parameter_settings():
    settings = list(ParameterGrid(RANDOM_FOREST_PARAMETER_GRID))
    if RF_MAX_SETTINGS is None or RF_MAX_SETTINGS >= len(settings):
        return settings

    rng = np.random.default_rng(SEED)
    chosen = np.sort(
        rng.choice(len(settings), size=RF_MAX_SETTINGS, replace=False)
    )
    return [settings[i] for i in chosen]

print("Random Forest settings to evaluate:", len(get_rf_parameter_settings()))

Random Forest settings to evaluate: 64


## 12. Evaluation helpers

The primary selection score is `macro_f1`.

The multiclass Brier score is calculated as the mean squared distance between the predicted probability vector and the one-hot observed outcome. Lower values are better.

Macro average precision and macro one-vs-rest ROC-AUC are supplementary discrimination summaries. They are calculated only when all required class conditions are satisfied.

In [14]:
def multiclass_brier_score(y_true, probabilities, n_classes=N_CLASSES):
    y_onehot = np.eye(n_classes)[np.asarray(y_true, dtype=int)]
    return float(np.mean(np.sum((probabilities - y_onehot) ** 2, axis=1)))


def evaluate_multiclass_probabilities(y_true, probabilities, predicted=None):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities)

    if predicted is None:
        predicted = probabilities.argmax(axis=1)

    p, r, f, support = precision_recall_fscore_support(
        y_true,
        predicted,
        labels=np.arange(N_CLASSES),
        zero_division=0,
    )

    result = {
        "accuracy": accuracy_score(y_true, predicted),
        "balanced_accuracy": balanced_accuracy_score(y_true, predicted),
        "macro_precision": precision_score(
            y_true, predicted, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            y_true, predicted, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_true, predicted, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, predicted, average="weighted", zero_division=0
        ),
        "log_loss": log_loss(
            y_true, probabilities, labels=np.arange(N_CLASSES)
        ),
        "brier_multiclass": multiclass_brier_score(y_true, probabilities),
    }

    y_bin = label_binarize(y_true, classes=np.arange(N_CLASSES))
    try:
        result["macro_average_precision_ovr"] = average_precision_score(
            y_bin, probabilities, average="macro"
        )
    except ValueError:
        result["macro_average_precision_ovr"] = np.nan

    try:
        result["macro_roc_auc_ovr"] = roc_auc_score(
            y_true,
            probabilities,
            multi_class="ovr",
            average="macro",
            labels=np.arange(N_CLASSES),
        )
    except ValueError:
        result["macro_roc_auc_ovr"] = np.nan

    for i, label in enumerate(CLASS_LABELS):
        result[f"precision_{label}"] = p[i]
        result[f"recall_{label}"] = r[i]
        result[f"f1_{label}"] = f[i]
        result[f"support_{label}"] = int(support[i])

    return result


def fit_pipeline_with_optional_weight(
    pipeline,
    X,
    y,
    sample_weight_mode="none",
):
    fit_kwargs = {}

    if sample_weight_mode == "balanced":
        weights = compute_sample_weight(class_weight="balanced", y=y)
        fit_kwargs["model__sample_weight"] = weights
    elif sample_weight_mode != "none":
        raise ValueError(f"Unknown sample_weight_mode: {sample_weight_mode}")

    return pipeline.fit(X, y, **fit_kwargs)

In [15]:
def evaluate_parameter_setting_cv(model_name, features, params):
    fold_rows = []

    params = dict(params)
    sample_weight_mode = params.pop("sample_weight_mode", "none")

    for spec in CV_FOLDS:
        fold_train_mask = (
            train_mask
            & severity_df["INCIDENT_YEAR"].isin(spec["train_years"])
        )
        fold_val_mask = (
            train_mask
            & severity_df["INCIDENT_YEAR"].isin(spec["validation_years"])
        )

        X_fold_train = severity_df.loc[fold_train_mask, features]
        y_fold_train = severity_df.loc[fold_train_mask, "SEVERITY_TARGET"]
        X_fold_val = severity_df.loc[fold_val_mask, features]
        y_fold_val = severity_df.loc[fold_val_mask, "SEVERITY_TARGET"]

        pipeline = make_pipeline(model_name, features)
        pipeline.set_params(**params)

        pipeline = fit_pipeline_with_optional_weight(
            pipeline,
            X_fold_train,
            y_fold_train,
            sample_weight_mode=sample_weight_mode,
        )

        probabilities = pipeline.predict_proba(X_fold_val)
        predicted = probabilities.argmax(axis=1)

        metrics = evaluate_multiclass_probabilities(
            y_fold_val, probabilities, predicted
        )
        metrics.update({
            "fold": spec["fold"],
            "model": model_name,
            "sample_weight_mode": sample_weight_mode,
        })
        fold_rows.append(metrics)

    return pd.DataFrame(fold_rows)


def summarize_cv_setting(model_name, feature_set_name, params, fold_results):
    row = {
        "model": model_name,
        "feature_set": feature_set_name,
        "params": json.dumps(params, sort_keys=True, default=str),
    }

    metric_columns = [
        "macro_f1", "macro_precision", "macro_recall",
        "weighted_f1", "balanced_accuracy",
        "log_loss", "brier_multiclass",
        "macro_average_precision_ovr", "macro_roc_auc_ovr",
        "precision_S+D", "recall_S+D", "f1_S+D",
    ]

    for metric in metric_columns:
        row[f"mean_{metric}"] = fold_results[metric].mean()
        row[f"std_{metric}"] = fold_results[metric].std(ddof=1)

    return row

## 13. Majority-class baseline

The baseline establishes how much the trained candidates improve over simply predicting the most frequent training severity category.

Probability metrics are also calculated using the class-prior probabilities learned from 1990–2018.

In [16]:
X_train_baseline = severity_df.loc[train_mask, EXTENDED_FEATURES]
y_train = severity_df.loc[train_mask, "SEVERITY_TARGET"]

X_validation_baseline = severity_df.loc[validation_mask, EXTENDED_FEATURES]
y_validation = severity_df.loc[validation_mask, "SEVERITY_TARGET"]

dummy = DummyClassifier(strategy="prior", random_state=SEED)
dummy.fit(X_train_baseline, y_train)

dummy_val_prob = dummy.predict_proba(X_validation_baseline)
dummy_val_pred = dummy_val_prob.argmax(axis=1)

baseline_metrics = evaluate_multiclass_probabilities(
    y_validation,
    dummy_val_prob,
    dummy_val_pred,
)
baseline_metrics.update({
    "model": "dummy_prior",
    "feature_set": "none",
})

display(pd.DataFrame([baseline_metrics]).round(4))

,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,log_loss,brier_multiclass,macro_average_precision_ovr,macro_roc_auc_ovr,precision_M,recall_M,f1_M,support_M,precision_M?,recall_M?,f1_M?,support_M?,precision_S+D,recall_S+D,f1_S+D,support_S+D,model,feature_set
0,0.1086,0.3333,0.0362,0.3333,0.0653,0.0213,1.2713,0.7961,0.3333,0.5,0.1086,1.0,0.1959,209,0.0,0.0,0.0,1587,0.0,0.0,0.0,129,dummy_prior,none


### Baseline interpretation placeholder

The majority/prior baseline learned from the 1990–2018 training period predicts Minor (`M`), which is the most common training class. On the 2019–2021 validation period, however, `M?` has become the dominant observed class. Consequently, the baseline achieves only 0.065 Macro-F1 and 0.333 balanced accuracy. Its recall for `M` is 1.00 because every observation is classified as Minor, but recall and F1 are 0 for both `M?` and `S+D`, including 0 severe-class recall and 0 severe-class F1. Overall accuracy is also only 10.9%, reflecting the substantial change in class distribution between the training and validation periods. This provides a deliberately weak reference: useful trained models should materially improve Macro-F1 and must recognize both the undetermined and severe classes rather than reproducing the historical majority-class pattern.

# 14. Hyperparameter tuning

The three model families are tuned in separate sections so runtime and failures are easier to diagnose. Each section saves its result immediately.

**Primary ranking:** highest mean chronological-CV Macro-F1.

**Tie/context checks:** severe-class F1/recall, Macro-F1 fold variability, multiclass Brier score, and log loss.

The final candidate is not selected from Macro-F1 mechanically if a numerically tiny gain comes with materially worse severe-class behavior or probability quality. Such a tradeoff must be documented explicitly.

## 14.1 Logistic Regression tuning

In [ ]:
logistic_tuning_rows = []

for feature_set_name, features in FEATURE_SETS.items():
    for params in ParameterGrid(LOGISTIC_PARAMETER_GRID):
        print("Logistic:", feature_set_name, params)

        fold_results = evaluate_parameter_setting_cv(
            "logistic", features, params
        )
        logistic_tuning_rows.append(
            summarize_cv_setting(
                "logistic", feature_set_name, params, fold_results
            )
        )

logistic_tuning_results = (
    pd.DataFrame(logistic_tuning_rows)
    .sort_values(
        ["mean_macro_f1", "mean_f1_S+D", "mean_brier_multiclass"],
        ascending=[False, False, True],
    )
)

logistic_tuning_results.to_csv(
    OUTPUT_DIR / "logistic_severity_chronological_cv_results.csv",
    index=False,
)
display(logistic_tuning_results.head(40).round(4))

### Logistic Regression tuning interpretation

> **[EXECUTION INFERENCE PLACEHOLDER]** Replace after execution. Report the selected feature set, `C`, solver/penalty, whether `class_weight="balanced"` was selected, mean Macro-F1 ± SD, severe-class precision/recall/F1, and probability-quality metrics. Explain whether weighting improved minority recognition and whether it carried a calibration/log-loss tradeoff.

## 14.2 Random Forest tuning

This is intentionally isolated because it is the slowest candidate family. `n_jobs=-1`, sparse one-hot preprocessing, the smaller damaged-only sample, and the optional reproducible parameter sampling are the runtime optimizations used here.

In [ ]:
random_forest_tuning_rows = []

rf_settings = get_rf_parameter_settings()

for feature_set_name, features in FEATURE_SETS.items():
    for params in rf_settings:
        print("Random Forest:", feature_set_name, params)

        fold_results = evaluate_parameter_setting_cv(
            "random_forest", features, params
        )
        random_forest_tuning_rows.append(
            summarize_cv_setting(
                "random_forest", feature_set_name, params, fold_results
            )
        )

random_forest_tuning_results = (
    pd.DataFrame(random_forest_tuning_rows)
    .sort_values(
        ["mean_macro_f1", "mean_f1_S+D", "mean_brier_multiclass"],
        ascending=[False, False, True],
    )
)

random_forest_tuning_results.to_csv(
    OUTPUT_DIR / "random_forest_severity_chronological_cv_results.csv",
    index=False,
)
display(random_forest_tuning_results.head(40).round(4))

### Random Forest tuning interpretation

> **[EXECUTION INFERENCE PLACEHOLDER]** Replace after execution. Record whether the full or sampled grid was used. Report the selected depth, tree count, leaf/split controls, feature sampling, and class-weight option. Compare its Macro-F1 and `S+D` behavior against Logistic Regression and note the runtime tradeoff.

## 14.3 XGBoost tuning

In [ ]:
xgboost_tuning_rows = []

for feature_set_name, features in FEATURE_SETS.items():
    for params in ParameterGrid(XGBOOST_PARAMETER_GRID):
        print("XGBoost:", feature_set_name, params)

        fold_results = evaluate_parameter_setting_cv(
            "xgboost", features, params
        )
        xgboost_tuning_rows.append(
            summarize_cv_setting(
                "xgboost", feature_set_name, params, fold_results
            )
        )

xgboost_tuning_results = (
    pd.DataFrame(xgboost_tuning_rows)
    .sort_values(
        ["mean_macro_f1", "mean_f1_S+D", "mean_brier_multiclass"],
        ascending=[False, False, True],
    )
)

xgboost_tuning_results.to_csv(
    OUTPUT_DIR / "xgboost_severity_chronological_cv_results.csv",
    index=False,
)
display(xgboost_tuning_results.head(40).round(4))

### XGBoost tuning interpretation

> **[EXECUTION INFERENCE PLACEHOLDER]** Replace after execution. Report the selected tree complexity, learning rate, sampling/regularization settings, and whether balanced `sample_weight` was selected. Compare Macro-F1, severe-class F1/recall, log loss, and Brier score against the other candidates.

## 14.4 Combined chronological-CV evidence and provisional selection

The best setting for each model/feature-set pair is selected by mean Macro-F1. The table intentionally keeps severe-class metrics and probability metrics alongside it.

The candidate carried forward should satisfy two goals simultaneously:

1. improve multiclass discrimination/classification over the baseline;
2. preserve enough probability quality that calibration has a realistic chance to support simulation.

A candidate with slightly higher Macro-F1 but substantially poorer `S+D` recall or markedly worse log loss/Brier score should be discussed rather than selected automatically.

In [ ]:
all_tuning_results = pd.concat(
    [
        logistic_tuning_results,
        random_forest_tuning_results,
        xgboost_tuning_results,
    ],
    ignore_index=True,
)

best_by_model_feature = (
    all_tuning_results
    .sort_values(
        ["mean_macro_f1", "mean_f1_S+D", "mean_brier_multiclass"],
        ascending=[False, False, True],
    )
    .groupby(["model", "feature_set"], as_index=False)
    .first()
    .sort_values(
        ["mean_macro_f1", "mean_f1_S+D"],
        ascending=False,
    )
)

display(best_by_model_feature.round(4))
best_by_model_feature.to_csv(
    OUTPUT_DIR / "severity_cv_selected_settings.csv",
    index=False,
)

provisional_best_row = best_by_model_feature.iloc[0]
print("Provisional best model:", provisional_best_row["model"])
print("Feature set:", provisional_best_row["feature_set"])
print("Parameters:", provisional_best_row["params"])

### Chronological-CV interpretation placeholder

> **[EXECUTION INFERENCE PLACEHOLDER]** Replace after execution. State the ranking of Logistic Regression, Random Forest, and XGBoost by mean Macro-F1. Include the `S+D` recall/F1 and probability metrics so the selection is not presented as a single-score decision. If the top two models are close, explain the tradeoff and retain both into 2019–2021 validation if appropriate.

## 15. Refit CV-selected candidates on 1990–2018 and evaluate on 2019–2021

The best configuration for each model/feature-set pair is refitted on the complete training period. The untouched 2022–2024 test remains locked.

Predictions are saved so calibration experiments do not require retuning.

In [ ]:
def parse_selected_params(param_text):
    return json.loads(param_text)


fitted_candidates = {}
validation_rows = []
validation_predictions = pd.DataFrame({
    "INDEX_NR": severity_df.loc[validation_mask, "INDEX_NR"].to_numpy(),
    "INCIDENT_YEAR": severity_df.loc[validation_mask, "INCIDENT_YEAR"].to_numpy(),
    "observed_severity": severity_df.loc[validation_mask, "SEVERITY_MODEL"].to_numpy(),
    "observed_severity_code": y_validation.to_numpy(),
})

for _, selected in best_by_model_feature.iterrows():
    model_name = selected["model"]
    feature_set_name = selected["feature_set"]
    features = FEATURE_SETS[feature_set_name]

    params = parse_selected_params(selected["params"])
    sample_weight_mode = params.pop("sample_weight_mode", "none")

    pipeline = make_pipeline(model_name, features)
    pipeline.set_params(**params)

    X_train = severity_df.loc[train_mask, features]
    X_val = severity_df.loc[validation_mask, features]

    pipeline = fit_pipeline_with_optional_weight(
        pipeline,
        X_train,
        y_train,
        sample_weight_mode=sample_weight_mode,
    )

    prob = pipeline.predict_proba(X_val)
    pred = prob.argmax(axis=1)

    metrics = evaluate_multiclass_probabilities(
        y_validation, prob, pred
    )
    candidate_name = f"{model_name}__{feature_set_name}"
    metrics.update({
        "candidate": candidate_name,
        "model": model_name,
        "feature_set": feature_set_name,
        "sample_weight_mode": sample_weight_mode,
    })
    validation_rows.append(metrics)
    fitted_candidates[candidate_name] = {
        "pipeline": pipeline,
        "features": features,
        "sample_weight_mode": sample_weight_mode,
        "selected_params": params,
    }

    for i, label in enumerate(CLASS_LABELS):
        validation_predictions[f"{candidate_name}__prob_{label}"] = prob[:, i]

    joblib.dump(
        pipeline,
        CANDIDATE_MODEL_DIR / f"severity_{candidate_name}.joblib",
    )

validation_results = (
    pd.DataFrame(validation_rows)
    .sort_values(
        ["macro_f1", "f1_S+D", "brier_multiclass"],
        ascending=[False, False, True],
    )
)

display(validation_results.round(4))

validation_results.to_csv(
    OUTPUT_DIR / "severity_2019_2021_candidate_metrics.csv",
    index=False,
)
validation_predictions.to_csv(
    OUTPUT_DIR / "severity_2019_2021_candidate_probabilities.csv",
    index=False,
)

### Validation interpretation placeholder

> **[EXECUTION INFERENCE PLACEHOLDER]** Replace after execution. Compare the CV ranking with 2019–2021 performance. Report Macro-F1, per-class precision/recall/F1, balanced accuracy, log loss, Brier score, and macro average precision. Identify any evidence of temporal degradation or a change in the `M?` class that affects model behavior.

## 16. Confusion matrices for validation candidates

Confusion matrices are retained even though the project is probability-first. They reveal whether a model systematically:

- confuses `S+D` with Minor;
- over-predicts severe damage;
- treats `M?` as a catch-all category.

Counts are shown directly rather than relying on accuracy.

In [ ]:
for candidate_name, candidate_info in fitted_candidates.items():
    features = candidate_info["features"]
    prob = candidate_info["pipeline"].predict_proba(
        severity_df.loc[validation_mask, features]
    )
    pred = prob.argmax(axis=1)

    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_validation,
        pred,
        labels=np.arange(N_CLASSES),
        display_labels=CLASS_LABELS,
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(f"2019–2021 validation — {candidate_name}")
    plt.tight_layout()
    plt.show()

## 17. Choose the base candidate before calibration

The base candidate is selected using 2019–2021 evidence, with Macro-F1 as the primary score and explicit consideration of:

- `S+D` precision, recall, and F1;
- log loss;
- multiclass Brier score;
- stability relative to chronological CV.

**Do not use 2022–2024 here.**

The automated default below selects the highest validation Macro-F1, breaking ties with `S+D` F1 and then Brier score. If the team overrides that rule for an academically defensible tradeoff, document the override in the markdown immediately below.

In [ ]:
selected_validation_row = validation_results.iloc[0]
SELECTED_CANDIDATE = selected_validation_row["candidate"]

print("Automatically selected base candidate:", SELECTED_CANDIDATE)
display(selected_validation_row.to_frame().T.round(4))

# Optional documented override:
# SELECTED_CANDIDATE = "xgboost__airport_aware"

### Base-candidate decision placeholder

> **[EXECUTION INFERENCE PLACEHOLDER]** Replace after execution. State the selected candidate and explain why. If the automatic Macro-F1 winner is overridden, provide a concrete metric-based reason (for example, a negligible Macro-F1 difference but substantially stronger `S+D` recall and/or probability quality).

# 18. Forward calibration experiment inside 2019–2021

Severity probabilities can enter simulation only if their reliability is evaluated.

To preserve temporal order:

- the selected base classifier remains fitted on 1990–2018;
- **2019** is used as the calibration-fitting period;
- **2020–2021** is used to compare raw, sigmoid, and isotonic calibration.

This is analogous to Notebook 06's forward calibration logic.

Calibration is performed one-vs-rest for each class and the resulting class scores are renormalized to sum to one. `M?` remains a nominal class.

The final test remains locked.

In [ ]:
calibration_fit_mask = severity_df["INCIDENT_YEAR"].eq(2019)
calibration_eval_mask = severity_df["INCIDENT_YEAR"].between(2020, 2021)

selected_info = fitted_candidates[SELECTED_CANDIDATE]
selected_pipeline = selected_info["pipeline"]
selected_features = selected_info["features"]

X_cal_fit = severity_df.loc[calibration_fit_mask, selected_features]
y_cal_fit = severity_df.loc[calibration_fit_mask, "SEVERITY_TARGET"].to_numpy()

X_cal_eval = severity_df.loc[calibration_eval_mask, selected_features]
y_cal_eval = severity_df.loc[calibration_eval_mask, "SEVERITY_TARGET"].to_numpy()

raw_cal_fit_prob = selected_pipeline.predict_proba(X_cal_fit)
raw_cal_eval_prob = selected_pipeline.predict_proba(X_cal_eval)

print("Calibration-fit rows (2019):", len(y_cal_fit))
print("Calibration-evaluation rows (2020–2021):", len(y_cal_eval))
print("2019 class counts:", np.bincount(y_cal_fit, minlength=N_CLASSES))
print("2020–2021 class counts:", np.bincount(y_cal_eval, minlength=N_CLASSES))

In [ ]:
class OneVsRestProbabilityCalibrator:
    def __init__(self, method="sigmoid"):
        if method not in {"sigmoid", "isotonic"}:
            raise ValueError("method must be 'sigmoid' or 'isotonic'")
        self.method = method
        self.models_ = []

    def fit(self, raw_probabilities, y_true):
        raw_probabilities = np.asarray(raw_probabilities)
        y_true = np.asarray(y_true, dtype=int)
        self.models_ = []

        for class_idx in range(raw_probabilities.shape[1]):
            binary_y = (y_true == class_idx).astype(int)
            x = raw_probabilities[:, class_idx]

            if self.method == "sigmoid":
                model = LogisticRegression(
                    solver="lbfgs",
                    random_state=SEED,
                )
                model.fit(x.reshape(-1, 1), binary_y)
            else:
                model = IsotonicRegression(
                    out_of_bounds="clip",
                    y_min=0.0,
                    y_max=1.0,
                )
                model.fit(x, binary_y)

            self.models_.append(model)

        return self

    def predict_proba(self, raw_probabilities):
        raw_probabilities = np.asarray(raw_probabilities)
        calibrated = np.zeros_like(raw_probabilities, dtype=float)

        for class_idx, model in enumerate(self.models_):
            x = raw_probabilities[:, class_idx]
            if self.method == "sigmoid":
                calibrated[:, class_idx] = model.predict_proba(
                    x.reshape(-1, 1)
                )[:, 1]
            else:
                calibrated[:, class_idx] = model.predict(x)

        calibrated = np.clip(calibrated, 1e-12, None)
        calibrated /= calibrated.sum(axis=1, keepdims=True)
        return calibrated


sigmoid_calibrator = OneVsRestProbabilityCalibrator("sigmoid").fit(
    raw_cal_fit_prob, y_cal_fit
)
isotonic_calibrator = OneVsRestProbabilityCalibrator("isotonic").fit(
    raw_cal_fit_prob, y_cal_fit
)

calibration_methods = {
    "raw": raw_cal_eval_prob,
    "sigmoid": sigmoid_calibrator.predict_proba(raw_cal_eval_prob),
    "isotonic": isotonic_calibrator.predict_proba(raw_cal_eval_prob),
}

calibration_rows = []
for method, prob in calibration_methods.items():
    row = evaluate_multiclass_probabilities(y_cal_eval, prob)
    row["calibration_method"] = method
    calibration_rows.append(row)

calibration_results = (
    pd.DataFrame(calibration_rows)
    .sort_values(
        ["brier_multiclass", "log_loss", "macro_f1"],
        ascending=[True, True, False],
    )
)

display(calibration_results.round(4))
calibration_results.to_csv(
    OUTPUT_DIR / "severity_forward_calibration_comparison.csv",
    index=False,
)

### Calibration decision rule

The calibration method is selected primarily by **multiclass Brier score and log loss** because this stage concerns probability reliability, not threshold classification.

Calibration is **not forced**. If raw probabilities are better than both calibration methods, the raw system is retained.

Macro-F1 is shown to confirm that calibration has not introduced unexpected argmax-classification behavior, but it is not the primary calibration criterion.

In [ ]:
SELECTED_CALIBRATION_METHOD = calibration_results.iloc[0]["calibration_method"]
print("Selected calibration method:", SELECTED_CALIBRATION_METHOD)

### Calibration interpretation placeholder

> **[EXECUTION INFERENCE PLACEHOLDER]** Replace after execution. Compare raw, sigmoid, and isotonic Brier score and log loss on 2020–2021. State whether calibration materially improves probability quality. If isotonic performs unusually well or poorly, discuss the available class support—especially `S+D`—rather than treating the result as automatically stable.

## 19. Class-wise calibration plots

These plots treat each severity class as one-vs-rest and compare predicted probability bins with observed class frequency.

They are diagnostic plots; they do not imply that `M`, `M?`, and `S+D` form an ordinal sequence.

In [ ]:
def plot_classwise_calibration(y_true, probability_dict, n_bins=8):
    y_true = np.asarray(y_true)

    for class_idx, label in enumerate(CLASS_LABELS):
        plt.figure(figsize=(6, 5))
        plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")

        binary_y = (y_true == class_idx).astype(int)

        for method, probs in probability_dict.items():
            p = probs[:, class_idx]
            bins = np.linspace(0, 1, n_bins + 1)
            ids = np.digitize(p, bins[1:-1], right=True)

            mean_pred = []
            observed = []
            for b in range(n_bins):
                m = ids == b
                if m.sum() == 0:
                    continue
                mean_pred.append(p[m].mean())
                observed.append(binary_y[m].mean())

            plt.plot(mean_pred, observed, marker="o", label=method)

        plt.xlabel(f"Predicted P({label})")
        plt.ylabel(f"Observed frequency of {label}")
        plt.title(f"Forward calibration — class {label}")
        plt.legend()
        plt.tight_layout()
        plt.show()

plot_classwise_calibration(y_cal_eval, calibration_methods)

# 20. Lock the final severity probability system

At this point, all decisions must be fixed **before opening 2022–2024**:

1. target definition: `M`, `M?`, `S+D`;
2. feature set;
3. model family;
4. hyperparameters;
5. class/sample weighting choice;
6. calibration method.

The selected base model remains trained on 1990–2018, and the calibrator is refitted using all 2019–2021 probabilities. This mirrors the role of the calibration period in Notebook 06.

In [ ]:
# Refit the selected calibrator using all 2019–2021 validation data.
X_validation_selected = severity_df.loc[validation_mask, selected_features]
y_validation_all = severity_df.loc[validation_mask, "SEVERITY_TARGET"].to_numpy()
raw_validation_all_prob = selected_pipeline.predict_proba(X_validation_selected)

if SELECTED_CALIBRATION_METHOD == "raw":
    final_calibrator = None
elif SELECTED_CALIBRATION_METHOD == "sigmoid":
    final_calibrator = OneVsRestProbabilityCalibrator("sigmoid").fit(
        raw_validation_all_prob, y_validation_all
    )
elif SELECTED_CALIBRATION_METHOD == "isotonic":
    final_calibrator = OneVsRestProbabilityCalibrator("isotonic").fit(
        raw_validation_all_prob, y_validation_all
    )
else:
    raise ValueError("Unknown calibration method.")

locked_configuration = {
    "target_classes": CLASS_LABELS,
    "destroyed_merge": "D merged into S for modelling only -> S+D",
    "m_question_mark": "retained as independent undetermined-level class",
    "selected_candidate": SELECTED_CANDIDATE,
    "selected_features": selected_features,
    "sample_weight_mode": selected_info["sample_weight_mode"],
    "selected_params": selected_info["selected_params"],
    "selected_calibration_method": SELECTED_CALIBRATION_METHOD,
    "train_period": "1990-2018",
    "calibration_period": "2019-2021",
    "final_test_period": "2022-2024",
    "seed": SEED,
}

with open(
    OUTPUT_DIR / "severity_locked_configuration.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(locked_configuration, f, indent=2, default=str)

display(pd.Series(locked_configuration, name="locked_value"))

# 21. FINAL TEST UNLOCK — 2022–2024

**Do not run this section until every cell above has completed and the locked configuration has been saved.**

The final test is used once to answer whether the severity component generalizes sufficiently for downstream use.

The decision must consider:

- overall Macro-F1;
- `S+D` precision, recall, and F1;
- confusion matrix;
- log loss and Brier score;
- calibration behavior;
- performance by test year;
- stability of the `M?` class;
- support limitations.

No hyperparameters, feature sets, or calibration methods may be changed in response to this final-test result.

In [ ]:
X_test = severity_df.loc[test_mask, selected_features]
y_test = severity_df.loc[test_mask, "SEVERITY_TARGET"].to_numpy()

raw_test_prob = selected_pipeline.predict_proba(X_test)

if final_calibrator is None:
    final_test_prob = raw_test_prob
else:
    final_test_prob = final_calibrator.predict_proba(raw_test_prob)

final_test_pred = final_test_prob.argmax(axis=1)

final_test_metrics = evaluate_multiclass_probabilities(
    y_test,
    final_test_prob,
    final_test_pred,
)

display(pd.DataFrame([final_test_metrics]).round(4))

pd.DataFrame([final_test_metrics]).to_csv(
    OUTPUT_DIR / "severity_final_test_metrics.csv",
    index=False,
)

test_predictions = pd.DataFrame({
    "INDEX_NR": severity_df.loc[test_mask, "INDEX_NR"].to_numpy(),
    "INCIDENT_YEAR": severity_df.loc[test_mask, "INCIDENT_YEAR"].to_numpy(),
    "observed_severity": severity_df.loc[test_mask, "SEVERITY_MODEL"].to_numpy(),
    "predicted_severity": [INT_TO_LABEL[i] for i in final_test_pred],
})

for i, label in enumerate(CLASS_LABELS):
    test_predictions[f"prob_{label}"] = final_test_prob[:, i]

test_predictions.to_csv(
    OUTPUT_DIR / "severity_final_test_predictions.csv",
    index=False,
)

### Final-test interpretation placeholder

> **[EXECUTION INFERENCE PLACEHOLDER]** Replace after execution. Report Macro-F1, balanced accuracy, per-class precision/recall/F1, macro average precision, log loss, and Brier score. Give particular attention to `S+D` false negatives and false positives. Compare with 2019–2021 validation and state whether the change suggests temporal degradation.

## 22. Final-test confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    final_test_pred,
    labels=np.arange(N_CLASSES),
    display_labels=CLASS_LABELS,
    cmap="Blues",
    colorbar=False,
    ax=ax,
)
ax.set_title("Severity model — locked 2022–2024 final test")
plt.tight_layout()
plt.show()

## 23. Final-test performance by year

An aggregate 2022–2024 score can hide drift. Metrics are therefore reported separately for each final-test year. These are descriptive diagnostics only and cannot be used to retune the model.

In [ ]:
year_rows = []

test_positions = severity_df.loc[test_mask, ["INCIDENT_YEAR"]].reset_index(drop=True)

for year in sorted(test_positions["INCIDENT_YEAR"].unique()):
    m = test_positions["INCIDENT_YEAR"].eq(year).to_numpy()

    metrics = evaluate_multiclass_probabilities(
        y_test[m],
        final_test_prob[m],
        final_test_pred[m],
    )
    metrics["year"] = int(year)
    metrics["rows"] = int(m.sum())
    year_rows.append(metrics)

year_metrics = pd.DataFrame(year_rows)
display(year_metrics.round(4))
year_metrics.to_csv(
    OUTPUT_DIR / "severity_final_test_metrics_by_year.csv",
    index=False,
)

### Year-by-year interpretation placeholder

> **[EXECUTION INFERENCE PLACEHOLDER]** Replace after execution. Describe whether Macro-F1, `S+D` recall/F1, and probability metrics are stable across 2022, 2023, and 2024. Where `S+D` support is small, emphasize the count and avoid overinterpreting year-specific percentages.

# 24. Known-severity sensitivity analysis: `M` versus `S+D`

`M?` creates a scientifically important complication: it represents an **unknown level of damage**, not a physical point on a severity continuum.

The main three-class model appropriately preserves this category. As a sensitivity analysis, this section asks a narrower question:

> **Among damaged incidents whose severity level was actually determined, can scenario-eligible predictors distinguish Minor from Substantial/Destroyed damage?**

This analysis:

- excludes `M?`;
- does not replace the main three-class model;
- must be described as a selected known-severity subset;
- is useful for distinguishing physical-severity predictability from predictability of reporting completeness.

To avoid doubling the full tuning burden, the sensitivity analysis uses the **already selected model family, feature set, and structural hyperparameters**. It is therefore diagnostic rather than a second independent model search.

In [ ]:
known_df = severity_df.loc[
    severity_df["SEVERITY_MODEL"].isin(["M", "S+D"])
].copy()

known_df["KNOWN_SEVERITY_TARGET"] = (
    known_df["SEVERITY_MODEL"].eq("S+D").astype(int)
)

known_train = known_df["INCIDENT_YEAR"].between(1990, 2018)
known_validation = known_df["INCIDENT_YEAR"].between(2019, 2021)
known_test = known_df["INCIDENT_YEAR"].between(2022, 2024)

known_summary = []
for split_name, mask in [
    ("train", known_train),
    ("validation", known_validation),
    ("test", known_test),
]:
    subset = known_df.loc[mask]
    known_summary.append({
        "split": split_name,
        "rows": len(subset),
        "minor": int((subset["KNOWN_SEVERITY_TARGET"] == 0).sum()),
        "severe_S_plus_D": int((subset["KNOWN_SEVERITY_TARGET"] == 1).sum()),
        "severe_rate": subset["KNOWN_SEVERITY_TARGET"].mean(),
    })

display(pd.DataFrame(known_summary).round(4))

### Known-severity sensitivity interpretation placeholder

> **[EXECUTION INFERENCE PLACEHOLDER]** After the main analysis is complete, optionally implement/evaluate the selected model family as a binary `M` vs `S+D` sensitivity model. Compare its severe-class discrimination with the main three-class model. If the known-severity model is much stronger, explain that part of the main model's difficulty may arise from the `M?` reporting/uncertainty category rather than physical severity alone.

# 25. Decision gate: RETAIN, SIMPLIFY, or DROP

There is no universal metric threshold that automatically proves severity is suitable for Monte Carlo simulation. The decision is evidence-based and should be conservative because downstream simulation would repeatedly sample these probabilities.

Use the following gate.

### RETAIN

Retain the three-class severity model only if:

- it materially exceeds the majority baseline;
- Macro-F1 is useful and reasonably stable from CV → validation → final test;
- `S+D` is not effectively ignored;
- class-specific precision/recall are interpretable with their support counts;
- Brier score/log loss and calibration plots show usable probability behavior;
- no severe temporal collapse appears in 2022–2024.

### SIMPLIFY

Simplify if the three-class model is weak primarily because `M?` cannot be predicted reliably, while a defensible narrower representation has materially better support. Possible downstream simplification must be documented explicitly; it must **not** silently reinterpret `M?` as Minor.

### DROP

Drop severity from simulation if:

- the final system is only marginally above baseline;
- `S+D` performance is inadequate or unstable;
- probabilities remain poorly calibrated;
- temporal generalization is weak;
- or the model would create more confidence than the data support.

Dropping severity does **not** mean Substantial/Destroyed damage is unimportant. It means the available FAA reporting data and approved scenario predictors are insufficient to estimate its conditional probability reliably enough for this project's simulation.

The primary Notebook 06 damage-probability model remains valid independently of this secondary decision.

In [ ]:
# Fill this in only after reviewing every final-test output.
SEVERITY_DECISION = "PENDING"  # Replace with: "RETAIN", "SIMPLIFY", or "DROP"

ALLOWED_DECISIONS = {"PENDING", "RETAIN", "SIMPLIFY", "DROP"}
assert SEVERITY_DECISION in ALLOWED_DECISIONS

print("Severity decision:", SEVERITY_DECISION)

### Final severity decision — execution placeholder

> **[EXECUTION INFERENCE PLACEHOLDER]**
>
> **Decision:** `PENDING`
>
> Replace this section after the locked final test. State **RETAIN**, **SIMPLIFY**, or **DROP**, and support the decision with:
>
> - CV Macro-F1 and variability;
> - 2019–2021 validation Macro-F1;
> - 2022–2024 Macro-F1;
> - `S+D` precision, recall, and F1;
> - Brier score and log loss before/after calibration;
> - evidence from calibration plots;
> - any meaningful temporal drift;
> - class-support limitations, especially for the severe category.
>
> Do not make the decision from accuracy alone.

## 26. Save a final severity artifact only if the gate passes

Candidate models have already been written to `models/candidates/`.

A canonical severity probability artifact is written to `models/final/` **only when `SEVERITY_DECISION == "RETAIN"`**. This prevents Notebook 10 or the interface from accidentally consuming an unapproved secondary model.

The artifact contains:

- fitted 1990–2018 base pipeline;
- selected feature list;
- target-label mapping;
- selected calibration method;
- fitted 2019–2021 calibrator;
- locked hyperparameters;
- modelling decision metadata.

In [ ]:
severity_artifact = {
    "base_pipeline": selected_pipeline,
    "features": selected_features,
    "class_labels": CLASS_LABELS,
    "label_to_int": LABEL_TO_INT,
    "int_to_label": INT_TO_LABEL,
    "calibration_method": SELECTED_CALIBRATION_METHOD,
    "calibrator": final_calibrator,
    "configuration": locked_configuration,
    "decision": SEVERITY_DECISION,
}

if SEVERITY_DECISION == "RETAIN":
    final_path = FINAL_MODEL_DIR / "final_severity_probability_system.joblib"
    joblib.dump(severity_artifact, final_path)
    print("Saved final severity system:", final_path)
else:
    print(
        "No final severity artifact written because decision is",
        SEVERITY_DECISION,
    )

with open(
    OUTPUT_DIR / "severity_decision.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "decision": SEVERITY_DECISION,
            "selected_candidate": SELECTED_CANDIDATE,
            "calibration_method": SELECTED_CALIBRATION_METHOD,
        },
        f,
        indent=2,
    )

## 27. Output inventory and hand-off

Expected persistent outputs:

### `outputs/severity/`

- `severity_raw_class_counts.csv`
- `severity_split_class_counts.csv`
- `severity_chronological_cv_design.csv`
- `logistic_severity_chronological_cv_results.csv`
- `random_forest_severity_chronological_cv_results.csv`
- `xgboost_severity_chronological_cv_results.csv`
- `severity_cv_selected_settings.csv`
- `severity_2019_2021_candidate_metrics.csv`
- `severity_2019_2021_candidate_probabilities.csv`
- `severity_forward_calibration_comparison.csv`
- `severity_locked_configuration.json`
- `severity_final_test_metrics.csv`
- `severity_final_test_predictions.csv`
- `severity_final_test_metrics_by_year.csv`
- `severity_decision.json`

### `models/candidates/`

CV-selected candidate severity pipelines.

### `models/final/`

`final_severity_probability_system.joblib` **only if the severity component receives a RETAIN decision**.

### Downstream rule

Notebook 10 must inspect `severity_decision.json` before attempting to load a severity model.

- `RETAIN` → simulation may sample calibrated severity conditional on damage.
- `SIMPLIFY` → Notebook 10 may use only the specifically approved simplified representation.
- `DROP` → severity remains descriptive and does not enter Monte Carlo.

## 28. Limitations

- This model is conditional on a **reported wildlife strike that already has `INDICATED_DAMAGE = 1`**.
- `M?` is a reporting-uncertainty class and not an ordered physical severity level.
- Combining `D` with `S` is a statistical-support compromise and does not erase their aviation-safety distinction.
- The FAA database is observational and subject to reporting-practice and completeness changes.
- Historical associations and model predictions are not causal effects.
- Class weighting changes the training objective and may affect probability calibration; this is why weighting is tuned and calibration is tested separately.
- Airport and category support can change over time.
- Very small subgroup counts can make apparently large metric differences unstable.
- A successful classification model is not automatically a trustworthy probability model; calibration evidence is required for simulation.

## 29. References

**Domain definitions**

- Federal Aviation Administration. *Wildlife Strike Database field/data dictionary*, damage-level definitions based on the ICAO IBIS Manual (Fourth Edition, 2001). The project copy is retained as the original data dictionary.

**Imbalanced classification**

- He, H., & Garcia, E. A. (2009). Learning from Imbalanced Data. *IEEE Transactions on Knowledge and Data Engineering, 21*(9), 1263–1284.

**Machine-learning implementations**

- Pedregosa, F., Varoquaux, G., Gramfort, A., et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.
- Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*.

**Probability/calibration caution**

- van den Goorbergh, R., van Smeden, M., Timmerman, D., & Van Calster, B. Research on class-imbalance corrections in risk-prediction models, showing that imbalance interventions can alter calibration. This is used here to justify validating weighted and unweighted candidates rather than assuming weighting is always beneficial.

### Project continuity

This notebook should be interpreted together with:

- Notebook 02 — canonical cleaning, feature eligibility, and split definitions;
- Notebook 05 — candidate-model and chronological hyperparameter-tuning design;
- Notebook 06 — probability calibration and final-test discipline;
- Notebook 10 — Monte Carlo simulation, which may consume severity only after this notebook's decision gate.